In [ ]:
!pip uninstall torch torchvision torchaudio ultralytics -y

In [ ]:
!pip install --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!pip install ultralytics

In [ ]:
!pip install roboflow

In [ ]:
!pip install load_dotenv

In [ ]:
import os
import ultralytics
from IPython.display import clear_output
from ultralytics import YOLO
import torch
from PIL import Image
from roboflow import Roboflow
import matplotlib.pyplot as plt

ultralytics.checks()
clear_output()

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Ninguna")

In [ ]:
DRIVE_URL = '/content/drive'
LOCAL_PATH = os.getcwd()

In [ ]:
def detect_environment():
    try:
        ipython_env = str(get_ipython())

        if 'google.colab' in ipython_env:
            return "Google Colab"
        elif 'zmqshell' in ipython_env:
            return "Jupyter Notebook Local"
        else:
            return "Terminal interactiva de Python"
    except NameError:
        return "Script de Python estándar (.py)"


print(f"Entorno detectado: {detect_environment()}")

In [ ]:
if detect_environment() != "Google Colab":
    BASE = LOCAL_PATH
else:
    BASE = DRIVE_URL
    from google.colab import drive
    drive.mount('/content/drive')
    from google.colab import userdata

In [ ]:
BASE_IMGS = os.path.join(BASE, "imgs")
BASE_MODELS = os.path.join(BASE, "models")
BASE_DATA = os.path.join(BASE, "data")
BASE_RESULTS = os.path.join(BASE, "results")
BASE_PROJECT = os.path.join(BASE_DATA, "bccd-1")

In [ ]:
os.makedirs(BASE_IMGS, exist_ok=True)
os.makedirs(BASE_MODELS, exist_ok=True)
os.makedirs(BASE_DATA, exist_ok=True)
os.makedirs(BASE_RESULTS, exist_ok=True)
os.makedirs(BASE_PROJECT, exist_ok=True)

In [ ]:
model_n= YOLO(os.path.join(BASE_MODELS, "yolov8n.pt"))

In [ ]:
result_n = model_n.predict(os.path.join(BASE_IMGS, "imagen_yolo.jpg"))

In [ ]:
result_n[0].names

In [ ]:
result_n[0].boxes

In [ ]:
for r in result_n:
    im_array = r.plot(line_width=2)
    im_rgb = im_array[...,::-1]
    im = Image.fromarray(im_rgb)
    plt.imshow(im_rgb)
    plt.axis('off')
    plt.show()
    im.save(os.path.join(BASE_IMGS, "detection_n.jpg"))

# Entrenamiento de Modelo

In [ ]:
model_s= YOLO(os.path.join(BASE_MODELS, "yolov8s.pt"))

In [ ]:
def setup_roboflow_credentials():
    env = detect_environment()
    api_key = None

    if env == "colab":
        try:
            api_key = userdata.get('ROBOFLOW_API_KEY')
            print("Credenciales de Roboflow cargadas desde Secretos de Colab.")
        except Exception:
            print("Configura 'ROBOFLOW_API_KEY' en los secretos de Colab.")
    else:
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            print("python-dotenv no está instalado. Usando variables de entorno del sistema...")

        api_key = os.environ.get('ROBOFLOW_API_KEY')

        if api_key:
            print("Credenciales de Roboflow cargadas desde variables de entorno (.env).")
        else:
            print("No se encontró 'ROBOFLOW_API_KEY' en el entorno local.")

    return api_key

In [ ]:
rf_api_key = setup_roboflow_credentials()

In [ ]:
if rf_api_key:
    try:
        rf = Roboflow(api_key=rf_api_key)

        workspace_name = 'yolov8-rdmbf'
        project_name = 'bccd-f0tcy'

        print(f"Descargando proyecto '{project_name}' del workspace '{workspace_name}'...")

        project = rf.workspace(workspace_name).project(project_name)
        version = project.version(1)
        os.chdir(BASE_DATA)
        dataset = version.download("yolov8")
        os.chdir(BASE)
        print("Dataset descargado correctamente en:", BASE_DATA)

    except Exception as e:
        print(f"Ocurrió un error al conectar con Roboflow: {e}")
else:
    print("Ejecución detenida: No hay credenciales válidas para Roboflow.")

In [ ]:
result= model_s.train(data=os.path.join(BASE_PROJECT, "data.yaml"), project= BASE_RESULTS, name='bccd_custom_training', epochs=10, imgsz=416, batch=8,device=0, amp=False, workers=4, cache=True, optimize='auto')

In [ ]:
result_l = model_s.predict(os.path.join(BASE_IMGS, "imagen_yolo.jpg"), save= True, show=False, conf=0.25)

In [ ]:
result_l[0].names

In [ ]:
result_l[0].boxes

In [ ]:
for r in result_l:
    im_array = r.plot(line_width=2)
    im_rgb = im_array[...,::-1]
    im = Image.fromarray(im_rgb)
    plt.imshow(im_rgb)
    plt.axis('off')
    plt.show()
    im.save(os.path.join(BASE_IMGS, "detection_s.jpg"))